# Snowy 2.0 IPS — IFC validator demonstration (v2)

This is v2 of the validator notebook. Changes from v1:

1. **All 8 schema rules now implemented** — R1, R2 (Bernoulli), R3, R4 (stage applicability), R5, R6, R7, R8.
2. **Synthetic IFC generation** — uses IfcOpenShell to produce a real `.ifc 4.3` file you can validate, view in BIM tools, and share with vendors.
3. **Microsecond timestamps** in audit-trail filenames so concurrent runs don't overwrite each other.
4. **POSIX paths** in JSON reports for cross-platform compatibility.
5. **Per-rule chart** that deduplicates multi-segment violations into one bar per rule.

**Files needed in the same folder as this notebook:**

```
ips_validator.py
generate_synthetic_ifc.py
```

**Optional dependencies:**
- `ifcopenshell` — required for cells 8-10 (real IFC generation and validation)
- `pandas`, `matplotlib` — for the visualisation cells

## 1. Setup

In [ ]:
import sys
from pathlib import Path

from ips_validator import IPSDictValidator, IPSValidator, Severity

print("Validator loaded.")
print(f"Severity levels: {[s.value for s in Severity]}")

## 2. Build the clean Snowy 2.0 IPS dataset

The dictionary below now includes a `LifecycleStage` tag on every segment (required by R4) and operating pressures that decline along the alignment (required by R2).

In [ ]:
clean_dataset = {
    "segments": [
        {
            "guid": "seg_1+250",
            "psets": {
                "Pset_HydraulicPerformance_IPS": {
                    "SegmentChainageStart": 1000.0,
                    "InternalDiameter": 6.5,
                    "InternalDiameter_Source": "DESIGN",
                    "DesignDischarge_Generating": 337.0,
                    "DesignDischarge_Source": "DESIGN",
                    "DesignDischarge_Pumping": 280.0,
                    "MeanFlowVelocity_Generating": 10.16,
                    "ManningRoughnessCoefficient": 0.012,
                    "Manning_Source": "DESIGN",
                    "HeadLoss_Segment": 7.1,
                    "OperatingPressure_Mean": 4.92e6,
                    "WaterDensity": 999.7,
                },
                "Pset_CompositeLining_IPS": {
                    "RockMass_GSI": 58.0, "Rock_Source": "FIELD_MONITORING",
                    "RockMass_DeformationModulus": 1.2e10,
                    "EinsteinSchwartz_LoadShareConcrete": 0.32,
                    "EinsteinSchwartz_LoadShareSteelLiner": 0.30,
                    "EinsteinSchwartz_LoadShareRockMass": 0.38,
                    "ES_Source": "FLAC3D",
                },
                "Pset_FatigueDamage_IPS": {
                    "MinerDamageRatio_Cumulative": 0.31,
                    "Miner_Source": "SURROGATE_DEEPONET",
                    "MinerDamageThreshold_Action": 0.5,
                    "MinerDamageThreshold_Critical": 1.0,
                },
                "Pset_SurrogatePrediction_IPS": {
                    "PredictionTimestamp": "2026-04-25T10:00:00+00:00",
                    "PredictedDamageRatio_Mean": 0.31,
                },
                "Pset_AssessmentMeta_IPS": {
                    "LifecycleStage": "OPERATION",
                    "AssessmentTimestamp": "2026-05-08T14:00:00+00:00",
                },
            },
        },
        {
            "guid": "seg_1+750",
            "psets": {
                "Pset_HydraulicPerformance_IPS": {
                    "SegmentChainageStart": 1500.0,
                    "InternalDiameter": 6.5,
                    "InternalDiameter_Source": "DESIGN",
                    "DesignDischarge_Generating": 337.0,
                    "DesignDischarge_Source": "DESIGN",
                    "MeanFlowVelocity_Generating": 10.16,
                    "ManningRoughnessCoefficient": 0.012,
                    "Manning_Source": "DESIGN",
                    "HeadLoss_Segment": 7.1,
                    "OperatingPressure_Mean": 4.92e6 - 7.1 * 999.7 * 9.81,
                    "WaterDensity": 999.7,
                },
                "Pset_CompositeLining_IPS": {
                    "RockMass_GSI": 62.0, "Rock_Source": "FIELD_MONITORING",
                    "RockMass_DeformationModulus": 1.5e10,
                    "EinsteinSchwartz_LoadShareConcrete": 0.30,
                    "EinsteinSchwartz_LoadShareSteelLiner": 0.28,
                    "EinsteinSchwartz_LoadShareRockMass": 0.42,
                    "ES_Source": "FLAC3D",
                },
                "Pset_FatigueDamage_IPS": {
                    "MinerDamageThreshold_Action": 0.5,
                    "MinerDamageThreshold_Critical": 1.0,
                },
                "Pset_AssessmentMeta_IPS": {
                    "LifecycleStage": "OPERATION",
                    "AssessmentTimestamp": "2026-05-08T14:00:00+00:00",
                },
            },
        },
        {
            "guid": "seg_2+250",
            "psets": {
                "Pset_HydraulicPerformance_IPS": {
                    "SegmentChainageStart": 2000.0,
                    "InternalDiameter": 6.5,
                    "InternalDiameter_Source": "DESIGN",
                    "DesignDischarge_Generating": 337.0,
                    "DesignDischarge_Source": "DESIGN",
                    "MeanFlowVelocity_Generating": 10.16,
                    "HeadLoss_Segment": 7.1,
                    "OperatingPressure_Mean": 4.92e6 - 2 * 7.1 * 999.7 * 9.81,
                    "WaterDensity": 999.7,
                },
                "Pset_CompositeLining_IPS": {
                    "EinsteinSchwartz_LoadShareConcrete": 0.31,
                    "EinsteinSchwartz_LoadShareSteelLiner": 0.29,
                    "EinsteinSchwartz_LoadShareRockMass": 0.40,
                    "ES_Source": "FLAC3D",
                },
                "Pset_FatigueDamage_IPS": {
                    "MinerDamageThreshold_Action": 0.5,
                    "MinerDamageThreshold_Critical": 1.0,
                },
                "Pset_AssessmentMeta_IPS": {
                    "LifecycleStage": "OPERATION",
                    "AssessmentTimestamp": "2026-05-08T14:00:00+00:00",
                },
            },
        },
    ],
    "joints": [
        {"guid": "joint_1500",
         "psets": {"Pset_FACSJoint_IPS": {
             "JointGUID_Upstream": "seg_1+250",
             "JointGUID_Downstream": "seg_1+750",
             "JointStationing": 1500.0}}},
        {"guid": "joint_2000",
         "psets": {"Pset_FACSJoint_IPS": {
             "JointGUID_Upstream": "seg_1+750",
             "JointGUID_Downstream": "seg_2+250",
             "JointStationing": 2000.0}}},
    ],
}
print(f"Clean dataset built: {len(clean_dataset['segments'])} segments, "
      f"{len(clean_dataset['joints'])} joints")

## 3. Run all 8 rules against the clean dataset

You should see 8 green checks — every rule from §14 of the schema specification.

In [ ]:
report = IPSDictValidator(clean_dataset).run_all()

s = report.summary()
print(f"Pass: {s['pass']}, Warn: {s['warn']}, Error: {s['error']}")
print()
for r in report.results:
    icon = {"pass": "[OK]", "info": "[i] ", "warn": "[!] ", "error": "[X] "}[r.severity.value]
    print(f"{icon} {r.rule_id} — {r.rule_name}")
    print(f"     {r.message}")

## 4. Inject violations targeting all 8 rules

Each modification triggers a specific rule:

- R1: discharge mismatch between adjacent segments
- R2: pressure profile inconsistent with declared head loss
- R3: load shares no longer sum to 1.0
- R4: operation-stage properties on a DESIGN-tagged segment
- R5: source field deleted while value retained
- R6: surrogate prediction backdated by 113 days
- R7: action threshold inverted with critical threshold
- R8: joint references non-existent downstream segment

In [ ]:
import copy

faulty_dataset = copy.deepcopy(clean_dataset)

# R1: discharge mismatch
faulty_dataset["segments"][1]["psets"]["Pset_HydraulicPerformance_IPS"]["DesignDischarge_Generating"] = 250.0

# R2: break pressure profile (no longer matches declared head loss)
faulty_dataset["segments"][0]["psets"]["Pset_HydraulicPerformance_IPS"]["OperatingPressure_Mean"] = 3.92e6

# R3: load shares break closure (sum to 1.08)
faulty_dataset["segments"][0]["psets"]["Pset_CompositeLining_IPS"]["EinsteinSchwartz_LoadShareConcrete"] = 0.40

# R4: tag seg_2+250 as DESIGN but populate operation-only fields on it
faulty_dataset["segments"][2]["psets"]["Pset_AssessmentMeta_IPS"]["LifecycleStage"] = "DESIGN"
faulty_dataset["segments"][2]["psets"]["Pset_FatigueDamage_IPS"]["MinerDamageRatio_Cumulative"] = 0.29
faulty_dataset["segments"][2]["psets"]["Pset_SurrogatePrediction_IPS"] = {
    "PredictionTimestamp": "2026-04-25T10:00:00+00:00",
    "PredictedDamageRatio_Mean": 0.29,
}

# R5: missing source provenance
del faulty_dataset["segments"][0]["psets"]["Pset_HydraulicPerformance_IPS"]["DesignDischarge_Source"]

# R6: stale surrogate
faulty_dataset["segments"][0]["psets"]["Pset_SurrogatePrediction_IPS"]["PredictionTimestamp"] = "2026-01-15T10:00:00+00:00"

# R7: invert thresholds
faulty_dataset["segments"][2]["psets"]["Pset_FatigueDamage_IPS"]["MinerDamageThreshold_Action"] = 1.0
faulty_dataset["segments"][2]["psets"]["Pset_FatigueDamage_IPS"]["MinerDamageThreshold_Critical"] = 0.8

# R8: dangling joint reference
faulty_dataset["joints"][0]["psets"]["Pset_FACSJoint_IPS"]["JointGUID_Downstream"] = "missing"

print("Eight rule violations injected.")

## 5. Run the validator on the faulty dataset

In [ ]:
faulty_report = IPSDictValidator(faulty_dataset).run_all()

s = faulty_report.summary()
print(f"Pass: {s['pass']}, Warn: {s['warn']}, Error: {s['error']}")
print()
for r in faulty_report.results:
    icon = {"pass": "[OK]", "info": "[i] ", "warn": "[!] ", "error": "[X] "}[r.severity.value]
    guid = f"  ({r.segment_guid})" if r.segment_guid else ""
    print(f"{icon} {r.rule_id} — {r.rule_name}{guid}")
    print(f"     {r.message}\n")

## 6. Action items as a pandas DataFrame

In [ ]:
try:
    import pandas as pd
    rows = [{"Rule": r.rule_id, "Severity": r.severity.value.upper(),
             "Check": r.rule_name, "Segment": r.segment_guid or "—",
             "Message": r.message} for r in faulty_report.results]
    df = pd.DataFrame(rows)
    actions = df[df["Severity"].isin(["WARN", "ERROR"])].reset_index(drop=True)
    print(f"Action items: {len(actions)}")
    display(actions)
except ImportError:
    print("pip install pandas to enable this view.")

## 7. Per-rule severity chart (the corrected version)

This is the v2 chart — one bar per *rule*, coloured by the worst severity that rule reported. Eight bars, one per schema rule.

In [ ]:
try:
    import matplotlib.pyplot as plt

    severity_order = {"pass": 0, "info": 1, "warn": 2, "error": 3}
    severity_per_rule = {}
    for r in faulty_report.results:
        cur = severity_per_rule.get(r.rule_id, "pass")
        if severity_order[r.severity.value] > severity_order[cur]:
            severity_per_rule[r.rule_id] = r.severity.value
        elif r.rule_id not in severity_per_rule:
            severity_per_rule[r.rule_id] = r.severity.value

    rule_ids = sorted(severity_per_rule.keys())
    severities = [severity_per_rule[r] for r in rule_ids]

    color_map = {"pass": "#639922", "info": "#888780",
                 "warn": "#EF9F27", "error": "#E24B4A"}
    colors = [color_map[s] for s in severities]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(rule_ids, [1] * len(rule_ids), color=colors, edgecolor="white")
    for i, sev in enumerate(severities):
        ax.text(i, 0.5, sev.upper(), ha="center", va="center",
                color="white", fontweight="bold", fontsize=10)
    ax.set_ylim(0, 1.2)
    ax.set_yticks([])
    ax.set_xlabel("Rule ID")
    ax.set_title("Faulty dataset — worst severity per rule",
                 fontsize=12, loc="left", pad=10)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("pip install matplotlib to enable this view.")

## 8. Generate a synthetic IFC 4.3 file (real `.ifc` on disk)

This produces a real IFC file you can:
- Open in any IFC viewer (Solibri, BIMcollab, BlenderBIM)
- Validate using the IfcOpenShell-backed `IPSValidator` class
- Share with vendors as a conformance test fixture

In [ ]:
try:
    from generate_synthetic_ifc import build_synthetic_ifc

    clean_summary = build_synthetic_ifc("snowy2_synthetic_clean.ifc",
                                        inject_errors=False)
    print("Clean IFC generated:")
    for k, v in clean_summary.items():
        if k != "segment_guids" and k != "joint_guids":
            print(f"  {k}: {v}")

    print()
    faulty_summary = build_synthetic_ifc("snowy2_synthetic_faulty.ifc",
                                         inject_errors=True)
    print("Faulty IFC generated:")
    for k, v in faulty_summary.items():
        if k != "segment_guids" and k != "joint_guids":
            print(f"  {k}: {v}")
except ImportError:
    print("pip install ifcopenshell to enable IFC generation.")

## 9. Validate the synthetic IFC files

This is the full end-to-end pipeline: synthetic generator → real `.ifc` 4.3 file → IfcOpenShell-backed validator → 8-rule report.

In [ ]:
try:
    print("=" * 60)
    print("CLEAN IFC")
    print("=" * 60)
    clean_real = IPSValidator("snowy2_synthetic_clean.ifc").run_all()
    s = clean_real.summary()
    print(f"Pass: {s['pass']}, Warn: {s['warn']}, Error: {s['error']}")
    for r in clean_real.results:
        icon = {"pass": "[OK]", "info": "[i] ", "warn": "[!] ", "error": "[X] "}[r.severity.value]
        print(f"{icon} {r.rule_id} — {r.message}")

    print()
    print("=" * 60)
    print("FAULTY IFC")
    print("=" * 60)
    faulty_real = IPSValidator("snowy2_synthetic_faulty.ifc").run_all()
    s = faulty_real.summary()
    print(f"Pass: {s['pass']}, Warn: {s['warn']}, Error: {s['error']}")
    for r in faulty_real.results:
        icon = {"pass": "[OK]", "info": "[i] ", "warn": "[!] ", "error": "[X] "}[r.severity.value]
        print(f"{icon} {r.rule_id} — {r.message}")
except (ImportError, FileNotFoundError) as e:
    print(f"Skipping: {e}")

## 10. Reusable wrapper with v2 corrections

Microsecond-precision filenames + POSIX-style paths in the JSON output, so the same script works on Windows, Linux and macOS without surprises.

In [ ]:
import json
from datetime import datetime, timezone

def validate_ips_submission(dataset_or_path, output_dir="./validation_runs",
                              strict=True):
    Path(output_dir).mkdir(exist_ok=True)
    if isinstance(dataset_or_path, dict):
        v = IPSDictValidator(dataset_or_path)
        label = "in_memory"
    else:
        v = IPSValidator(str(dataset_or_path))
        label = Path(dataset_or_path).stem

    report = v.run_all()
    s = report.summary()

    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    report_path = Path(output_dir) / f"{label}_{ts}.json"
    with open(report_path, "w") as f:
        json.dump(report.to_dict(), f, indent=2, default=str)

    if s["error"] > 0:
        verdict = "REJECTED"
    elif strict and s["warn"] > 0:
        verdict = "REJECTED"
    elif s["warn"] > 0:
        verdict = "CONDITIONAL"
    else:
        verdict = "ACCEPTED"

    return {
        "verdict": verdict,
        "errors": s["error"], "warnings": s["warn"], "passes": s["pass"],
        "report_path": report_path.as_posix(),
    }

print("Clean dataset:")
print(f"  {validate_ips_submission(clean_dataset)}")
print()
print("Faulty dataset (strict):")
print(f"  {validate_ips_submission(faulty_dataset, strict=True)}")
print()
print("Warning-only dataset (strict vs non-strict):")
warn_only = copy.deepcopy(clean_dataset)
del warn_only["segments"][0]["psets"]["Pset_HydraulicPerformance_IPS"]["DesignDischarge_Source"]
print(f"  strict=True:  {validate_ips_submission(warn_only, strict=True)}")
print(f"  strict=False: {validate_ips_submission(warn_only, strict=False)}")

## What's new in v2 — recap

- **Eight rules instead of six.** Full coverage of the §14 schema specification.
- **Real IFC 4.3 files.** The synthetic generator gives you `.ifc` files for vendor handover and viewer inspection.
- **Microsecond timestamps** in audit-trail filenames prevent silent overwrites under tight loops.
- **POSIX paths** in JSON reports work the same on Windows, Linux and macOS.
- **Per-rule chart** — one bar per rule, deduplicated, with the worst severity surfaced.

## What to try next

Open `snowy2_synthetic_clean.ifc` in BlenderBIM or Solibri to see the property sets attached to the alignment segments. Edit values directly in the viewer, save, re-run cell 9 to confirm the validator catches your edits. This is the realistic loop a designer or asset operator would follow.